# Silver: Store — Bronze → Silver Cleansing

Operations performed:
- Deduplicate
- Trim / case normalisation
- Type casting
- Null handling
- Incremental load via `ingest_ts` watermark

In [0]:
dbutils.widgets.dropdown(name="environment", defaultValue="dev",
                         choices=["dev","qa","prd"], label="select Environment")
env = dbutils.widgets.get("environment")


In [0]:
silverTablName = f"saleslake_{env}.silver_{env}.cleanedStore"
bronzeTablName = f"saleslake_{env}.bronze_{env}.rawStore"
print(f"Source : {bronzeTablName}")
print(f"Target : {silverTablName}")


In [0]:
spark.sql(f"""
INSERT INTO {silverTablName}
SELECT DISTINCT
    CAST(TRIM(store_id) AS INT)               AS store_id,
    UPPER(TRIM(store_code))                   AS store_code,
    TRIM(store_name)                          AS store_name,
    UPPER(TRIM(store_type))                   AS store_type,
    TRIM(address)                             AS address,
    UPPER(TRIM(city))                         AS city,
    UPPER(TRIM(state))                        AS state,
    CAST(TRIM(region_id) AS INT)              AS region_id,
    TRIM(manager_name)                        AS manager_name,
    TO_DATE(TRIM(opening_date), 'yyyy-MM-dd') AS opening_date,
    CAST(TRIM(square_feet) AS INT)            AS square_feet,
    UPPER(TRIM(status))                       AS status,
    CURRENT_TIMESTAMP() AS ingest_ts
FROM {bronzeTablName}
WHERE ingest_ts > (
    SELECT COALESCE(MAX(ingest_ts), TO_TIMESTAMP("1990-01-01"))
    FROM {silverTablName}
)
ORDER BY store_id
""")

print(f"Silver load complete for {silverTablName}")
spark.sql(f"SELECT COUNT(*) AS row_count FROM {silverTablName}").display()
